# Download data for 2003 and 2012

In [ ]:
# Standard library
import os
import glob
import math
import random
import calendar
from pathlib import Path
import zipfile

# Scientific stack
import numpy as np
import pandas as pd
import xarray as xr
from scipy import stats
from scipy.stats import gaussian_kde

# Machine learning / compute
import dask
import sklearn

# Geospatial
import geopandas as gpd
from shapely.geometry import Point
from shapely.prepared import prep
from cartopy.io import shapereader
import cartopy.crs as ccrs
import cartopy.feature as cfeature


# APIs
import cdsapi


ModuleNotFoundError: No module named 'utci_nn_model3'

## 2003

### ERA5HEAT

In [ ]:

def download_thermal_comfort_data(year=2003, months=[6, 7, 8, 9], 
                                   output_dir='../data/2003_heatwave/utci_era5',
                                   variables=['universal_thermal_climate_index', 'mean_radiant_temperature']):
    """
    Download Thermal comfort indices from ERA5 reanalysis.
    
    Parameters:
    -----------
    year : int
        Year to download (default: 2003)
    months : list
        List of months to download (default: [6,7,8,9] for Jun-Sep)
    output_dir : str
        Directory to save output files
    variables : list
        List of variables to download. Options:
        - 'universal_thermal_climate_index' (UTCI)
        - 'mean_radiant_temperature' (MRT)
    """
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize the CDS API client
    c = cdsapi.Client()
    
    # Define the European region
    # Format: [North, West, South, East]
    europe_area = [72, -25, 35, 45]  # Covers most of Europe
    
    # Hours for the entire day (hourly data)
    hours = [f'{h:02d}:00' for h in range(24)]
    
    # Variable name mapping for file naming
    var_abbrev = {
        'universal_thermal_climate_index': 'utci',
        'mean_radiant_temperature': 'mrt'
    }
    
    print(f"Starting download for {year}, months: {months}")
    print(f"Variables: {[var_abbrev.get(v, v) for v in variables]}")
    print(f"Region: Europe {europe_area}")
    print("=" * 60)
    
    for month in months:
        month_name = calendar.month_abbr[month]
        
        # Get number of days in the month
        num_days = calendar.monthrange(year, month)[1]
        days = [f'{d:02d}' for d in range(1, num_days + 1)]
        
        for variable in variables:
            var_short = var_abbrev.get(variable, variable)
            output_file = f'{output_dir}/{var_short}_{year}_{month:02d}_{month_name}.nc'
            
            print(f"\nDownloading {var_short.upper()} for {month_name} {year}...")
            print(f"Days: 1-{num_days}")
            print(f"Output: {output_file}")
            
            try:
                c.retrieve(
                    'derived-utci-historical',
                    {
                        'version': '1_1',
                        'format': 'netcdf',
                        'variable': variable,
                        'product_type': 'consolidated_dataset',
                        'year': str(year),
                        'month': f'{month:02d}',
                        'day': days,
                        'time': hours,
                        'area': europe_area,  # [North, West, South, East]
                    },
                    output_file
                )
                print(f"✓ Successfully downloaded {var_short.upper()} for {month_name} {year}")
                
            except Exception as e:
                print(f"✗ Error downloading {var_short.upper()} for {month_name} {year}: {str(e)}")
                continue
    
    print("\n" + "=" * 60)
    print("Download complete!")


# Main execution
    # Download both UTCI and MRT
download_thermal_comfort_data(
    year=2003,
    months=[6, 7, 8, 9],  # June to September
    output_dir='../data/2003_heatwave/utci_era5',
    variables=['universal_thermal_climate_index', 'mean_radiant_temperature']
)


### ERA5 input

In [ ]:
def download_era5_single_level_data(year=2003, months=[6, 7, 8, 9], 
                                     output_dir='../data/2003_heatwave/era5_single_level',
                                     variables=['10m_u_component_of_wind', 
                                               '10m_v_component_of_wind',
                                               '2m_dewpoint_temperature',
                                               '2m_temperature']):
    """
    Download ERA5 hourly single level data from 1940 to present.
    
    Parameters:
    -----------
    year : int
        Year to download (default: 2003)
    months : list
        List of months to download (default: [6,7,8,9] for Jun-Sep)
    output_dir : str
        Directory to save output files
    variables : list
        List of variables to download. Options:
        - '10m_u_component_of_wind'
        - '10m_v_component_of_wind'
        - '2m_dewpoint_temperature'
        - '2m_temperature'
    """
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize the CDS API client
    c = cdsapi.Client()
    
    # Define the European region
    # Format: [North, West, South, East]
    europe_area = [72, -25, 35, 45]  # Covers most of Europe
    
    # Hours for the entire day (hourly data)
    hours = [f'{h:02d}:00' for h in range(24)]
    
    # Variable name mapping for file naming
    var_abbrev = {
        '10m_u_component_of_wind': 'u10',
        '10m_v_component_of_wind': 'v10',
        '2m_dewpoint_temperature': 'd2m',
        '2m_temperature': 't2m'
    }
    
    print(f"Starting download for {year}, months: {months}")
    print(f"Variables: {[var_abbrev.get(v, v) for v in variables]}")
    print(f"Region: Europe {europe_area}")
    print("=" * 60)
    
    for month in months:
        month_name = calendar.month_abbr[month]
        
        # Get number of days in the month
        num_days = calendar.monthrange(year, month)[1]
        days = [f'{d:02d}' for d in range(1, num_days + 1)]
        
        for variable in variables:
            var_short = var_abbrev.get(variable, variable)
            output_file = f'{output_dir}/{var_short}_{year}_{month:02d}_{month_name}.nc'
            
            print(f"\nDownloading {var_short.upper()} for {month_name} {year}...")
            print(f"Days: 1-{num_days}")
            print(f"Output: {output_file}")
            
            try:
                c.retrieve(
                    'reanalysis-era5-single-levels',
                    {
                        'product_type': 'reanalysis',
                        'format': 'netcdf',
                        'variable': variable,
                        'year': str(year),
                        'month': f'{month:02d}',
                        'day': days,
                        'time': hours,
                        'area': europe_area,  # [North, West, South, East]
                    },
                    output_file
                )
                print(f"✓ Successfully downloaded {var_short.upper()} for {month_name} {year}")
                
            except Exception as e:
                print(f"✗ Error downloading {var_short.upper()} for {month_name} {year}: {str(e)}")
                continue
    
    print("\n" + "=" * 60)
    print("Download complete!")



# Download all four variables
download_era5_single_level_data(
    year=2003,
    months=[6, 7, 8, 9],  # June to September
    output_dir='../data/2003_heatwave/input_era5',
    variables=['10m_u_component_of_wind', 
                '10m_v_component_of_wind',
                '2m_dewpoint_temperature',
                '2m_temperature']
    )
    


### Extract the UTCI and MRT data from zip files

In [ ]:
# Extract all the .nc files
data_dir = '../data/2003_heatwave/utci_era5'

for filename in os.listdir(data_dir):
    if filename.endswith('.nc'):
        filepath = os.path.join(data_dir, filename)
        
        print(f"Extracting {filename}...")
        
        # Extract the zip file
        with zipfile.ZipFile(filepath, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
            print(f"  Extracted: {zip_ref.namelist()}")

print("\nDone! Now listing all files:")
for file in os.listdir(data_dir):
    print(f"  {file}")

Extracting mrt_2003_06_Jun.nc...
  Extracted: ['ECMWF_mrt_20030601_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030602_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030603_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030604_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030605_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030606_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030607_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030608_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030609_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030610_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030611_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030612_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030613_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030614_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030615_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20030616_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mr

## 2012 Cold Spell

### Download Di Napoli UTCI and MRT

In [ ]:

def download_thermal_comfort_data(year=2003, months=[6, 7, 8, 9], 
                                   output_dir='../data/2003_heatwave/utci_era5',
                                   variables=['universal_thermal_climate_index', 'mean_radiant_temperature']):
    """
    Download Thermal comfort indices from ERA5 reanalysis.
    
    Parameters:
    -----------
    year : int
        Year to download (default: 2003)
    months : list
        List of months to download (default: [6,7,8,9] for Jun-Sep)
    output_dir : str
        Directory to save output files
    variables : list
        List of variables to download. Options:
        - 'universal_thermal_climate_index' (UTCI)
        - 'mean_radiant_temperature' (MRT)
    """
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize the CDS API client
    c = cdsapi.Client()
    
    # Define the European region
    # Format: [North, West, South, East]
    europe_area = [72, -25, 35, 45]  # Covers most of Europe
    
    # Hours for the entire day (hourly data)
    hours = [f'{h:02d}:00' for h in range(24)]
    
    # Variable name mapping for file naming
    var_abbrev = {
        'universal_thermal_climate_index': 'utci',
        'mean_radiant_temperature': 'mrt'
    }
    
    print(f"Starting download for {year}, months: {months}")
    print(f"Variables: {[var_abbrev.get(v, v) for v in variables]}")
    print(f"Region: Europe {europe_area}")
    print("=" * 60)
    
    for month in months:
        month_name = calendar.month_abbr[month]
        
        # Get number of days in the month
        num_days = calendar.monthrange(year, month)[1]
        days = [f'{d:02d}' for d in range(1, num_days + 1)]
        
        for variable in variables:
            var_short = var_abbrev.get(variable, variable)
            output_file = f'{output_dir}/{var_short}_{year}_{month:02d}_{month_name}.nc'
            
            print(f"\nDownloading {var_short.upper()} for {month_name} {year}...")
            print(f"Days: 1-{num_days}")
            print(f"Output: {output_file}")
            
            try:
                c.retrieve(
                    'derived-utci-historical',
                    {
                        'version': '1_1',
                        'format': 'netcdf',
                        'variable': variable,
                        'product_type': 'consolidated_dataset',
                        'year': str(year),
                        'month': f'{month:02d}',
                        'day': days,
                        'time': hours,
                        'area': europe_area,  # [North, West, South, East]
                    },
                    output_file
                )
                print(f"✓ Successfully downloaded {var_short.upper()} for {month_name} {year}")
                
            except Exception as e:
                print(f"✗ Error downloading {var_short.upper()} for {month_name} {year}: {str(e)}")
                continue
    
    print("\n" + "=" * 60)
    print("Download complete!")


# Main execution
    # Download both UTCI and MRT
download_thermal_comfort_data(
    year=2012,
    months=[1,2],  # Jan and Feb
    output_dir='../data/2012_coldspell/utci_era5',
    variables=['universal_thermal_climate_index', 'mean_radiant_temperature']
)


2025-11-20 17:21:14,808 WARNING [2025-09-26T00:00:00] The download of tropical nights statistics has been temporarily suspended. We are currently investigating the source of some unexpected values in these fields. At this time, we are unable to provide a specific timeframe for resolving this issue.  We apologize for the inconvenience and appreciate your patience while we address this matter.
2025-11-20 17:21:14,809 INFO Request ID is be127fba-47da-4446-817a-1d6e053b48e2


Starting download for 2012, months: [1, 2]
Variables: ['utci', 'mrt']
Region: Europe [72, -25, 35, 45]

Days: 1-31
Output: ../data/2012_coldspell/utci_era5/utci_2012_01_Jan.nc


2025-11-20 17:21:14,870 INFO status has been updated to accepted
2025-11-20 17:21:36,167 WARNING Download format not supported for this dataset. Defaulting to zip.
2025-11-20 17:21:36,168 INFO status has been updated to running
2025-11-20 17:23:09,216 INFO status has been updated to successful
2025-11-20 17:23:14,284 WARNING [2025-09-26T00:00:00] The download of tropical nights statistics has been temporarily suspended. We are currently investigating the source of some unexpected values in these fields. At this time, we are unable to provide a specific timeframe for resolving this issue.  We apologize for the inconvenience and appreciate your patience while we address this matter.
2025-11-20 17:23:14,286 INFO Request ID is f11c8ff6-c7b4-4013-a370-564fbfdb5d5d


✓ Successfully downloaded UTCI for Jan 2012

Days: 1-31
Output: ../data/2012_coldspell/utci_era5/mrt_2012_01_Jan.nc


2025-11-20 17:23:14,354 INFO status has been updated to accepted
2025-11-20 17:23:27,897 WARNING Download format not supported for this dataset. Defaulting to zip.
2025-11-20 17:23:27,899 INFO status has been updated to running
2025-11-20 17:24:29,910 INFO status has been updated to successful
2025-11-20 17:24:35,881 WARNING [2025-09-26T00:00:00] The download of tropical nights statistics has been temporarily suspended. We are currently investigating the source of some unexpected values in these fields. At this time, we are unable to provide a specific timeframe for resolving this issue.  We apologize for the inconvenience and appreciate your patience while we address this matter.
2025-11-20 17:24:35,883 INFO Request ID is 49525126-c0e7-45cb-8d85-a4dacfdb439a


✓ Successfully downloaded MRT for Jan 2012

Days: 1-29
Output: ../data/2012_coldspell/utci_era5/utci_2012_02_Feb.nc


2025-11-20 17:24:35,957 INFO status has been updated to accepted
2025-11-20 17:24:44,458 INFO status has been updated to running
2025-11-20 17:24:49,576 WARNING Download format not supported for this dataset. Defaulting to zip.
2025-11-20 17:25:51,548 INFO status has been updated to successful
2025-11-20 17:25:56,547 WARNING [2025-09-26T00:00:00] The download of tropical nights statistics has been temporarily suspended. We are currently investigating the source of some unexpected values in these fields. At this time, we are unable to provide a specific timeframe for resolving this issue.  We apologize for the inconvenience and appreciate your patience while we address this matter.
2025-11-20 17:25:56,550 INFO Request ID is aa4a95da-f6bf-41d0-a86b-7b5b3e47af79


✓ Successfully downloaded UTCI for Feb 2012

Days: 1-29
Output: ../data/2012_coldspell/utci_era5/mrt_2012_02_Feb.nc


2025-11-20 17:25:56,619 INFO status has been updated to accepted
2025-11-20 17:26:10,213 WARNING Download format not supported for this dataset. Defaulting to zip.
2025-11-20 17:26:10,214 INFO status has been updated to running
2025-11-20 17:27:12,160 WARNING Download format not supported for this dataset. Defaulting to zip.
2025-11-20 17:28:49,260 INFO status has been updated to successful
                                                                                          

✓ Successfully downloaded MRT for Feb 2012

Download complete!


### Download ERA5 input variables

In [ ]:
def download_era5_single_level_data(year=2003, months=[6, 7, 8, 9], 
                                     output_dir='../data/2003_heatwave/era5_single_level',
                                     variables=['10m_u_component_of_wind', 
                                               '10m_v_component_of_wind',
                                               '2m_dewpoint_temperature',
                                               '2m_temperature']):
    """
    Download ERA5 hourly single level data from 1940 to present.
    
    Parameters:
    -----------
    year : int
        Year to download (default: 2003)
    months : list
        List of months to download (default: [6,7,8,9] for Jun-Sep)
    output_dir : str
        Directory to save output files
    variables : list
        List of variables to download. Options:
        - '10m_u_component_of_wind'
        - '10m_v_component_of_wind'
        - '2m_dewpoint_temperature'
        - '2m_temperature'
    """
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Initialize the CDS API client
    c = cdsapi.Client()
    
    # Define the European region
    # Format: [North, West, South, East]
    europe_area = [72, -25, 35, 45]  # Covers most of Europe
    
    # Hours for the entire day (hourly data)
    hours = [f'{h:02d}:00' for h in range(24)]
    
    # Variable name mapping for file naming
    var_abbrev = {
        '10m_u_component_of_wind': 'u10',
        '10m_v_component_of_wind': 'v10',
        '2m_dewpoint_temperature': 'd2m',
        '2m_temperature': 't2m'
    }
    
    print(f"Starting download for {year}, months: {months}")
    print(f"Variables: {[var_abbrev.get(v, v) for v in variables]}")
    print(f"Region: Europe {europe_area}")
    print("=" * 60)
    
    for month in months:
        month_name = calendar.month_abbr[month]
        
        # Get number of days in the month
        num_days = calendar.monthrange(year, month)[1]
        days = [f'{d:02d}' for d in range(1, num_days + 1)]
        
        for variable in variables:
            var_short = var_abbrev.get(variable, variable)
            output_file = f'{output_dir}/{var_short}_{year}_{month:02d}_{month_name}.nc'
            
            print(f"\nDownloading {var_short.upper()} for {month_name} {year}...")
            print(f"Days: 1-{num_days}")
            print(f"Output: {output_file}")
            
            try:
                c.retrieve(
                    'reanalysis-era5-single-levels',
                    {
                        'product_type': 'reanalysis',
                        'format': 'netcdf',
                        'variable': variable,
                        'year': str(year),
                        'month': f'{month:02d}',
                        'day': days,
                        'time': hours,
                        'area': europe_area,  # [North, West, South, East]
                    },
                    output_file
                )
                print(f"✓ Successfully downloaded {var_short.upper()} for {month_name} {year}")
                
            except Exception as e:
                print(f"✗ Error downloading {var_short.upper()} for {month_name} {year}: {str(e)}")
                continue
    
    print("\n" + "=" * 60)
    print("Download complete!")



# Download all four variables
download_era5_single_level_data(
    year=2012,
    months=[1,2],  #Jan and Feb
    output_dir='../data/2012_coldspell/input_era5',
    variables=['10m_u_component_of_wind', 
                '10m_v_component_of_wind',
                '2m_dewpoint_temperature',
                '2m_temperature']
    )
    


Starting download for 2012, months: [1, 2]
Variables: ['u10', 'v10', 'd2m', 't2m']
Region: Europe [72, -25, 35, 45]

Days: 1-31
Output: ../data/2012_coldspell/input_era5/u10_2012_01_Jan.nc


2025-11-20 17:30:48,521 INFO Request ID is 2f8c734c-bf20-4fc2-b6d2-6ddbdf0f7374
2025-11-20 17:30:48,597 INFO status has been updated to accepted
2025-11-20 17:30:57,058 INFO status has been updated to running
2025-11-20 17:32:42,685 INFO status has been updated to successful


✓ Successfully downloaded U10 for Jan 2012

Days: 1-31
Output: ../data/2012_coldspell/input_era5/v10_2012_01_Jan.nc


2025-11-20 17:33:17,187 INFO Request ID is d2765d76-39a0-491f-b4c3-30cfb4130640
2025-11-20 17:33:17,271 INFO status has been updated to accepted
2025-11-20 17:33:30,837 INFO status has been updated to running
2025-11-20 17:35:11,316 INFO status has been updated to successful


✓ Successfully downloaded V10 for Jan 2012

Days: 1-31
Output: ../data/2012_coldspell/input_era5/d2m_2012_01_Jan.nc


...

✓ Successfully downloaded V10 for Feb 2012

Days: 1-29
Output: ../data/2012_coldspell/input_era5/d2m_2012_02_Feb.nc


2025-11-20 17:43:11,312 INFO Request ID is a9964980-a778-4995-98f6-84dda3d9c072
2025-11-20 17:43:11,419 INFO status has been updated to accepted
2025-11-20 17:43:25,072 INFO status has been updated to running
2025-11-20 17:44:27,071 INFO status has been updated to successful


✓ Successfully downloaded D2M for Feb 2012

Days: 1-29
Output: ../data/2012_coldspell/input_era5/t2m_2012_02_Feb.nc


2025-11-20 17:44:46,748 INFO Request ID is 54ba5ae9-fe58-477b-99b4-b3299df74673
2025-11-20 17:44:46,814 INFO status has been updated to accepted
2025-11-20 17:45:00,375 INFO status has been updated to running
2025-11-20 17:46:02,424 INFO status has been updated to successful
                                                                                         

✓ Successfully downloaded T2M for Feb 2012

Download complete!


### Extract the UTCI and MRT data from zip files

In [ ]:
# Extract all the .nc files
data_dir = '../data/2012_coldspell/utci_era5'

for filename in os.listdir(data_dir):
    if filename.endswith('.nc'):
        filepath = os.path.join(data_dir, filename)
        
        print(f"Extracting {filename}...")
        
        # Extract the zip file
        with zipfile.ZipFile(filepath, 'r') as zip_ref:
            zip_ref.extractall(data_dir)
            print(f"  Extracted: {zip_ref.namelist()}")

print("\nDone! Now listing all files:")
for file in os.listdir(data_dir):
    print(f"  {file}")

Extracting mrt_2012_01_Jan.nc...
  Extracted: ['ECMWF_mrt_20120101_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120102_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120103_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120104_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120105_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120106_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120107_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120108_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120109_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120110_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120111_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120112_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120113_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120114_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120115_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mrt_20120116_v1.1_con.area-subset.72.45.35.-25.nc', 'ECMWF_mr